## Import bibliotek

In [13]:
import requests
from bs4 import BeautifulSoup

Potencjalne produkty

32918774
83177636
91869341

## Pobranie z serwisu [Ceneo.pl](https;//Ceneo.pl) opinii o wybranym produkcie

### Wysłanie żądania dostępu do zasobu (strony WWW)


In [14]:
headers = {
    "Cookie":"sv3=1.0_edd5e284-1e04-11f1-8f6d-7d844dec98d7; urdsc=1; userCeneo=ID=3e5d9c2e-d5a8-445a-ad18-85d9a99424c3; __RequestVerificationToken=wFVm9U6qAdyOgHlXyKkTC6juq6IT-pSA8hKPmqgKfLazrHLvwjFC3I2o2CVkbNjRC0zoDF65daikg7eeVSMv1zk6As_7w19szya5ptI-TM41; st2=_gd%3dwww.google.com%2csref%3dhttps%3a%2f%2fwww.google.com%2f%2c_t%3d63908914594%2cencode%3dtrue; ai_user=f7pQ8|2026-03-12T11:16:35.304Z; ai_session=9UJiv|1773314195468|1773315993773; __utmf=0951e77eacc681f9c32401b00333ffe2_Dsgqi6QMc9CtX7buqOpcIw%3D%3D; appType=%7B…ble_cookie=1; _ttp=01KKGW7T7JSWD4KTY9RFT104VN_.tt.1; ttcsid_CNK74OBC77U1PP7E4UR0=1773314238707::YCwGadZDcQ4xjZus9mTI.1.1773315994818.1; ttcsid=1773314238707::z2GVjvtWfOtn7lM29-uc.1.1773315994818.0; __gads=ID=923d0bcddead1efb:T=1773314237:RT=1773314820:S=ALNI_MZjbmxEbPJwrSqwych9DGiJBc2u9w; __gpi=UID=00001379719b9094:T=1773314237:RT=1773314820:S=ALNI_Mb-cOgiKO8IMxrLcJVbKWcX41D5Vw; __eoi=ID=6362c2d756a84f8f:T=1773314237:RT=1773314820:S=AA-AfjYZEU88Cv9HTl3Wm3XpARcQ; nps3=SessionStartTime=1773315992,SurveyId=67",
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:148.0) Gecko/20100101 Firefox/148.0",
    "Host":"www.ceneo.pl"
}

In [21]:
url = "https://www.ceneo.pl/124893467#tab=reviews"
response = requests.get(url)
print(response.status_code)

200


### Parsowanie strony z opiniami

In [16]:
page_dom = BeautifulSoup(response.text,"html.parser") # obiekt klasy BeautifulSoup zwraca document object model, to jest drzewo po którym można chodzić(za pomocą selektorów CSS)
print(page_dom)


<!DOCTYPE html>

<!--[if lt IE 7]> <html class="no-js lt-ie9 lt-ie8 lt-ie7  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if IE 7]> <html class="no-js lt-ie9 lt-ie8  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if IE 8]> <html class="no-js lt-ie9  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js non-direct ab-inactive" lang="pl">
<!--<![endif]-->
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<link href="https://image.ceneo.pl" rel="preconnect"/>
<link href="https://image.ceneostatic.pl" rel="preconnect"/>
<link href="https://www.google-analytics.com" rel="preconnect"/>
<link href="https://www.googleadservices.com" rel="preconnect"/>
<link href="https://adservice.google.com" rel="preconnect"/>
<link href="https://www.google.pl" rel="preconnect"/>
<link href="https://www.google.com" rel="preconnect"/>
<link href="https://cdn.ampproject.org" rel="preconnect"/>
<title>Urządzenie wielofunkcyj

In [23]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [18]:
opinion = page_dom.select_one("div.js_product-review")
print(type(opinion))

<class 'bs4.element.Tag'>


### Analiza struktury pojedynczej opinii

|składowa|nazwa|selektor|
|--------|-----|--------|
|opinia|opinion|div.js_product-review|div.js_product-review|
|identyfikator|opinion_id|["data-entry-id"]|
|autor|author|span.user-post__author-name|
|treść|content|div.user-post__text|
|ocena|score|span.user-post__score-count|
|rekomendacja|recommendation|span.user-post__author-recomendation > em|
|lista zalet|pros|div.review-feature__item--positive|
|lista wad|cons|div.review-feature__item--negative|
|dla ilu przydatna|thumbs_up|button.vote-yes > span|
|dla ilu nieprzydatna|thumbs_down|button.vote-no > span|
|data zamieszczenia|post_date|span.user-post__published > time:nth-child(1)[datetime]|
|data zakupu|purchase_date|span.user-post__published > time:nth-child(2)[datetime]|

In [24]:
opinion = opinions.pop(0) #  żeby nie brac przeglądu AI

In [25]:
all_opinions = []

for opinion in opinions:
    opinion_data = {}
    # identyfikator opinii
    opinion_data["opinion_id"] = opinion.get("data-entry-id")

    # Autor
    opinion_data["author"] = opinion.select_one("span.user-post__author-name").get_text(strip = True)
    
    # Treść opinii
    opinion_data["content"] = opinion.select_one("div.user-post__text").get_text(strip = True)

    # Ocena
    opinion_data["score"] = opinion.select_one("span.user-post__score-count").get_text(strip = True)
    opinion_data["score"] = float(opinion_data['score'].split('/')[0].replace(',', '.'))
    
    # Rekomendacja (Polecam/Nie polecam)
    try:
        opinion_data['recommendation'] = opinion.select_one('span.user-post__author-recomendation > em').get_text(strip = True)
    except AttributeError:
        opinion_data['recommendation'] = None
    
    # Lista zalet
    opinion_data["pros"] = [p.get_text(strip = True) for p in opinion.select('div.review-feature__item--positive')]
    
    # Lista wad
    opinion_data["cons"] = [c.get_text(strip = True) for c in opinion.select('div.review-feature__item--negative')]
    
    # Dla ilu osób przydatna
    opinion_data["useful"] = int(opinion.select_one("button.vote-yes > span").get_text(strip = True))
    
    # Dla ilu osób nieprzydatna
    opinion_data["unuseful"] = int(opinion.select_one("button.vote-no > span").get_text(strip = True))
    
    # Data zamieszczenia
    opinion_data["published_date"] = opinion.select_one('span.user-post__published > time:nth-child(1)').get('datetime')
    
    # Data zakupu
    try:
        opinion_data["purchase_date"] = opinion.select_one('span.user-post__published > time:nth-child(1)').get('datetime')
    except AttributeError:
        opinion_data['purchase_date'] = None 
    
    all_opinions.append(opinion_data)
    
# Wyświetlenie zebranych opinii
for single_opinion in all_opinions:
    print(single_opinion)

{'opinion_id': '19353234', 'author': 'e...k', 'content': 'Mała kompaktowa drukarka. Jakość wydruku bardzo dobra, nie za głośna, całkiem szybka. Mam kłopot z połączeniem z WIFI ale jest opcja drukowania z kabla USB więc na razie mi to nie przeszkadza. Zajmuje mało miejsca na biurku, a nieużywaną można postawić w pionie i jest jeszcze więcej miejsca.', 'score': 5.0, 'recommendation': 'Polecam', 'pros': [], 'cons': [], 'useful': 0, 'unuseful': 0, 'published_date': '2025-01-07 10:17:01', 'purchase_date': '2025-01-07 10:17:01'}
{'opinion_id': '13777122', 'author': 'Kreoger', 'content': 'Zalety:\n- kompaktowa i elegancka\n- jakość druku jak najbardziej okej\n- oprogramowanie w j.polskim \n- można drukować przez WiFi oraz Epson Share\n- dobrze trzyma na baterii\n- prosta w obsłudze\n- bardzo duża ilość formatów do druku\nWady:\n- szybkość druku nie jest porywająca\n- cena', 'score': 5.0, 'recommendation': 'Polecam', 'pros': ['dodatkowy zbiornik na zużyty atrament', 'głośność pracy', 'jakość w

In [ ]:
page = 1
next = True


while next:

    url = f"https://www.ceneo.pl/124893467#tab=reviews-{page}"
    response = requests.get(url, headers = headers)
    page_dom = BeautifulSoup(response.text, "html.parser")
    opinions = page_dom.select('div.js_product-review:not(.user-post-highlight)')
    print(f"{url} --> {response.status_code}")
    next = True if page_dom.select_one('button.pagination__next') else False
    page += 1